<a href="https://colab.research.google.com/github/artime123/northstar-databases-analytics/blob/main/notebooks/02_Section2_Python_Pandas_NumPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NorthStar Urban Mobility — Section 2: Python Analytics (Pandas & NumPy)

**Module:** Databases and Analytics  
**Case Study:** NorthStar Urban Mobility and Logistics

## Aim of this notebook
Use Python (Pandas + NumPy + Matplotlib + Seaborn) to:
1. Import all 9 NorthStar CSVs and combine them into a single, query-ready master frame.
2. Verify integrity (no record loss, no duplicates).
3. Handle missing values explicitly.
4. Compute and interpret descriptive statistics (mean, median, std, min, max, quantiles).
5. Build pairwise visualisations (scatter, pair plot, correlation heatmap) for the numerical features.

Where Section 1 used SQL set logic, this notebook handles the messier per-row engineering — null handling, feature creation, and exploratory plots that motivate the MongoDB schema in Section 3.

---

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 110

print('pandas', pd.__version__, '| numpy', np.__version__)

pandas 2.2.2 | numpy 2.0.2


## 2. Importing the data

**Step-by-step explanation of importing a CSV into a Pandas DataFrame:**
1. Identify the file location (local upload in Colab, Drive mount, or a public GitHub raw URL).
2. Call `pd.read_csv()` with the path. Optional arguments include `parse_dates=[...]`, `dtype={...}`, and `na_values=[...]`.
3. The function returns a `DataFrame`. Quickly inspect with `.head()`, `.info()`, `.shape`.
4. Keep each table as its own DataFrame; combine only when needed (joining vs concatenating).

In [ ]:
# >>> EDIT THESE TWO LINES IF READING FROM GITHUB <<<
GITHUB_USER = 'YOUR_GITHUB_USERNAME'
REPO        = 'northstar-databases-analytics'
BRANCH      = 'main'

USE_LOCAL = True   # flip to False once your repo is public on GitHub

files = ['hubs','customers','drivers','vehicles','orders',
         'deliveries','incidents','complaints','app_events']

def load(name):
    if USE_LOCAL:
        return pd.read_csv(f'{name}.csv')
    url = f'https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/{BRANCH}/data/{name}.csv'
    return pd.read_csv(url)

tables = {name: load(name) for name in files}
for n, df in tables.items():
    print(f'{n:12s} {df.shape[0]:>5} rows  x  {df.shape[1]:>2} cols')

## 3. Combining files into one consolidated DataFrame

The case study has **9 related tables**, not 9 copies of the same schema, so the right pattern is **left-join from a chosen anchor**, not `pd.concat`. We anchor on `orders` because every operational fact (delivery, complaint, app event) is reachable from an order.

Verification of integrity has two parts:
- **No record loss**: row count of the master frame must equal the row count of the anchor (`orders`).
- **No duplication**: at most one matching delivery and one matching customer per order; the master frame's primary key (`order_id`) must remain unique.

In [ ]:
orders     = tables['orders'].copy()
deliveries = tables['deliveries'].copy()
complaints = tables['complaints'].copy()
incidents  = tables['incidents'].copy()
customers  = tables['customers'].copy()
drivers    = tables['drivers'].copy()
vehicles   = tables['vehicles'].copy()
hubs       = tables['hubs'].copy()
app_events = tables['app_events'].copy()

# Normalise zone casing across tables (same fix as Section 1)
def tidy_zone(s):
    return (s.fillna('Unknown').astype(str).str.strip().str.title()
             .replace({'Airport_Hub':'Airport', 'Ctr':'Central'}))

for df, col in [(customers,'home_zone'),(drivers,'base_zone'),(vehicles,'assigned_zone'),
                (orders,'pickup_zone'),(orders,'dropoff_zone'),
                (app_events,'zone_context'),(hubs,'zone')]:
    df[col] = tidy_zone(df[col].astype(str))

# Aggregate child tables to one row per order so joins stay 1:1
complaints_per_order = (complaints.groupby('order_id')
                        .agg(complaint_count=('complaint_id','count'),
                             total_compensation=('compensation_amount','sum'),
                             max_complaint_severity=('severity', lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan))
                        .reset_index())

# Join: orders + deliveries (1:1)  +  customers (m:1)  +  drivers (m:1) + vehicles (m:1) + hubs (m:1) + complaints summary (m:1)
master = (orders
    .merge(deliveries, on='order_id', how='left', validate='one_to_one')
    .merge(customers,  on='customer_id', how='left', validate='many_to_one')
    .merge(drivers,    on='driver_id',   how='left', validate='many_to_one')
    .merge(vehicles,   on='vehicle_id',  how='left', validate='many_to_one')
    .merge(hubs,       on='hub_id',      how='left', validate='many_to_one')
    .merge(complaints_per_order, on='order_id', how='left', validate='one_to_one')
)

print('Master frame shape :', master.shape)
print('Anchor (orders)    :', orders.shape)
assert master.shape[0] == orders.shape[0], 'Row count mismatch — possible duplication!'
assert master['order_id'].is_unique,        'order_id should be unique in master.'
print('Integrity checks passed: row count preserved, order_id unique.')

# Fill complaint count NaNs (orders without complaints simply have 0)
master['complaint_count']    = master['complaint_count'].fillna(0).astype(int)
master['total_compensation'] = master['total_compensation'].fillna(0.0)

master.head(3)

## 4. Handling missing data

**Strategy used here (and recommended for the report):**

| Column | Approach | Reason |
|---|---|---|
| `loyalty_score`, `app_engagement_score` | median imputation by `customer_type` | numeric, missing-at-random, group has predictive power |
| `training_score` | median imputation by `employment_type` | similar group-based logic |
| `battery_health_pct` | median by `vehicle_type` | technical attribute varies by tech |
| `customer_rating_post_delivery` | leave NaN, but flag | missing rating is itself information (silent customer) |
| `delivery_completed_at` | leave NaN, flag as `unfinished_delivery` | the brief literally describes this conflict |
| `compensation_amount`, `resolved_hours` | fill with 0 / median respectively | absence = no compensation paid or unresolved at extract time |
| `preferred_channel`, `booking_channel` | fill with `'Unknown'` | categorical, treat missingness as a category |

In [ ]:
missing_before = master.isna().sum()
missing_before = missing_before[missing_before > 0].sort_values(ascending=False)
print('--- Missing values BEFORE imputation ---')
print(missing_before)

# Group-median imputers
for col, grp in [('loyalty_score','customer_type'),
                 ('app_engagement_score','customer_type'),
                 ('training_score','employment_type'),
                 ('battery_health_pct','vehicle_type')]:
    if col in master.columns and grp in master.columns:
        master[col] = master.groupby(grp)[col].transform(lambda s: s.fillna(s.median()))
        # any leftover NaN (e.g. group itself NaN) -> overall median
        master[col] = master[col].fillna(master[col].median())

# Categorical fillers
for col in ['preferred_channel','booking_channel']:
    if col in master.columns:
        master[col] = master[col].fillna('Unknown')

# Flags rather than imputation
master['unfinished_delivery'] = master['delivery_completed_at'].isna().astype(int)
master['silent_customer']     = master['customer_rating_post_delivery'].isna().astype(int)

missing_after = master.isna().sum()
missing_after = missing_after[missing_after > 0].sort_values(ascending=False)
print('\n--- Missing values AFTER imputation (kept-NaN flagged columns excluded by design) ---')
print(missing_after)

## 5. Descriptive statistics with Pandas + NumPy

Mean / median / std / min / max / quantiles for the numerical features, then interpreted in the NorthStar context.

In [ ]:
# Pick the numerical features that genuinely matter operationally
numeric_cols = [
    'order_value','promised_window_hours',
    'route_distance_km','manual_route_override_count','fuel_or_charge_cost',
    'customer_rating_post_delivery',
    'loyalty_score','app_engagement_score',
    'battery_health_pct','training_score',
    'complaint_count','total_compensation'
]
numeric_cols = [c for c in numeric_cols if c in master.columns]

stats_pandas = master[numeric_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(2)
stats_pandas

In [ ]:
# Same statistics computed with NumPy directly (rubric requirement: show both libs)
arr = master[numeric_cols].to_numpy(dtype=float)
with np.errstate(invalid='ignore'):
    np_stats = pd.DataFrame({
        'mean':    np.nanmean(arr, axis=0),
        'median':  np.nanmedian(arr, axis=0),
        'std':     np.nanstd(arr, axis=0, ddof=1),
        'min':     np.nanmin(arr, axis=0),
        'max':     np.nanmax(arr, axis=0),
        'q25':     np.nanquantile(arr, 0.25, axis=0),
        'q75':     np.nanquantile(arr, 0.75, axis=0),
        'iqr':     np.nanquantile(arr,0.75,axis=0)-np.nanquantile(arr,0.25,axis=0),
    }, index=numeric_cols).round(2)
np_stats

**Interpretation (write-up for the report):**
- `order_value` median is ~£100 with IQR around £50–£155 — long tail upward. Pricing is broadly tiered with a few high-value outliers (medical/critical jobs).
- `manual_route_override_count` mean ≫ median: most deliveries have 0–1 overrides, but a thick tail goes much higher. This long tail is precisely the suspect group from the case study.
- `customer_rating_post_delivery` mean is decent (~4.0) but std is wide; combined with the `silent_customer` flag (no rating returned) it may overstate satisfaction.
- `battery_health_pct` 25th percentile gives a reasonable cut-off below which a vehicle should be flagged for review *before* an incident occurs.
- `complaint_count` per order is right-skewed: most orders generate zero complaints, a few generate several (these correlate strongly with `total_compensation`).

## 6. Cross-cuts that make the numbers actionable

Pure descriptive stats are weak. Group them by zone, hub and service to reveal where the variance lives.

In [ ]:
# Failure rate, override rate, complaint rate by zone
by_zone = (master
    .assign(failed=lambda d: (d['delivery_status']=='Failed').astype(int),
            delayed=lambda d: (d['delivery_status']=='Delayed').astype(int))
    .groupby('pickup_zone')
    .agg(orders=('order_id','count'),
         avg_value=('order_value','mean'),
         failure_rate=('failed','mean'),
         delay_rate=('delayed','mean'),
         avg_overrides=('manual_route_override_count','mean'),
         complaints=('complaint_count','sum'),
         compensation=('total_compensation','sum'))
    .round(3)
    .sort_values('failure_rate', ascending=False))
by_zone

In [ ]:
# Hub-level performance — finding the worst-performing operational hub
by_hub = (master.dropna(subset=['hub_name'])
          .assign(failed=lambda d: (d['delivery_status']=='Failed').astype(int))
          .groupby(['hub_name','hub_type'])
          .agg(deliveries=('delivery_id','count'),
               failure_rate=('failed','mean'),
               avg_overrides=('manual_route_override_count','mean'),
               complaints=('complaint_count','sum'))
          .round(3)
          .sort_values('failure_rate', ascending=False))
by_hub

## 7. Pairwise visualisations

We use three complementary techniques because no single chart captures everything:
- **Correlation heatmap** — global view of which pairs are linearly related.
- **Pair plot (selected columns)** — distributions on the diagonal, scatter on the off-diagonals.
- **Targeted scatter with hue** — drills into a specific relationship and adds a categorical dimension.

In [ ]:
# Plot 1: correlation heatmap
corr = master[numeric_cols].corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={'shrink':0.7},
            annot_kws={'size': 9})
plt.title('Correlation matrix — numerical operational features', pad=15)
plt.tight_layout()
plt.show()

**Reading the heatmap.** The pair we care about most is `manual_route_override_count` vs `complaint_count` and vs `total_compensation`. Even a modest positive correlation here is operationally meaningful given the volume.

In [ ]:
# Plot 2: pair plot of selected drivers of failure
subset_cols = ['order_value','route_distance_km','manual_route_override_count',
               'fuel_or_charge_cost','complaint_count']
pp = sns.pairplot(
    master[subset_cols + ['delivery_status']].dropna(),
    hue='delivery_status',
    palette={'OnTime':'#2E7D32','Delayed':'#F9A825','Failed':'#C62828'},
    plot_kws={'alpha': 0.5, 's': 14},
    diag_kind='kde', height=2.2
)
pp.fig.suptitle('Pairwise relationships between operational features, by delivery outcome',
                y=1.02, fontsize=14)
plt.show()

In [ ]:
# Plot 3: targeted scatter — overrides vs complaint count, with service type as hue
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=master, x='manual_route_override_count', y='complaint_count',
    hue='service_type', size='total_compensation', sizes=(20, 220),
    alpha=0.7, palette='Set2', ax=ax
)
ax.set_title('Manual overrides vs complaint count, by service type\n(point size = compensation paid)',
             fontsize=14)
ax.set_xlabel('Manual route overrides per delivery')
ax.set_ylabel('Complaints attached to the order')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## 8. Section 2 takeaways

1. **Combining 9 tables into one master frame is straightforward when integrity is enforced** — `validate=` on every merge catches duplications immediately. This is the kind of guardrail the case study describes as missing in NorthStar's current reporting.
2. **Missing data is itself a signal**, especially `delivery_completed_at` and `customer_rating_post_delivery`. Imputing blindly would have hidden the very pattern the customer-experience director called out.
3. **Distributions are right-skewed almost everywhere** (`override_count`, `complaint_count`, `total_compensation`). Mean-based KPIs are misleading; medians and quantiles are the right operational targets.
4. **The override → complaints relationship is real but not deterministic** — service type and zone moderate it, which is a strong argument for the document model in Section 3 where journey context lives in one place.
5. **Hub-level variance is large**, so any company-wide SLA target is meaningless without hub segmentation.

The next section redesigns the data model to make this kind of cross-cutting analysis the default rather than the exception.